Run these examples from the complete `tcia_utils_v4` folder; see [setup and preview-release instructions](README.md).

# Controlled public metadata to a manifest handoff

A controlled inventory is public metadata, not authorization to retrieve a controlled payload. This example makes bounded metadata calls and writes a route-specific Data Retriever manifest plus an accounting companion in a temporary directory. It does not run Data Retriever, authenticate, or transfer data. The fixture inventory is synthetic.

In [ ]:
from importlib.metadata import version
from _support import MODE, SHORT_TITLE, DOI, make_client, checked
print("SYNTHETIC FIXTURE MODE — results below are examples, not live TCIA evidence" if MODE == "fixture"
      else "LIVE METADATA MODE — bounded calls to the configured TCIA service")
print("tcia_utils", version("tcia_utils"))

In [ ]:
from pathlib import Path
import os
import tempfile
controlled_title = "EXAMPLE" if MODE == "fixture" else "CMB-PCA"
with make_client() as tcia:
    datasets = checked("controlled dataset inventory", lambda: tcia.search_controlled_datasets(limit=5))
    if datasets:
        print("bounded controlled datasets", datasets.data["count"], datasets.completeness.state)
    files = checked("controlled file inventory", lambda: tcia.list_controlled_files(
        controlled_title, limit=2 if MODE == "fixture" else 1))
    if files:
        print("public file metadata rows", files.data["count"], "with DRS URI",
              sum(bool(row.get("drs_uri")) for row in files.data["files"]))
        print("authorization reminder", files.data["warning"])
        selection = tcia.select(files.data["files"], release_identity=files.provenance,
                                query_scope={"short_title": controlled_title,
                                             "record_grain": "file", "completeness": "unknown"})
        destination = Path(os.environ.get("TCIA_EXAMPLE_OUTPUT_DIR") or tempfile.mkdtemp(prefix="tcia-v4-manifest-"))
        report = tcia.export_selection(selection, destination)
        print("manifest files", [path.name for path in report.files])
        print("accounting", {"exported": report.exported, "deduplicated": report.deduplicated,
                             "excluded": report.excluded, "unresolved": report.unresolved})

Each CSV has exactly one recognized route column, such as `drs_uri`. The companion records unresolved rows and unknown sizes. An absent DRS URI stays unresolved. An actual controlled transfer requires a separate authorization workflow and is outside this notebook.